Code generated by Gemini.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Load datasets
df_train = pd.read_json('/content/drive/path/beeradvocate_train.json', lines=True)
df_val = pd.read_json('/content/drive/path/beeradvocate_val.json', lines=True)
df_test = pd.read_json('/content/drive/path/beeradvocate_test.json', lines=True)
print("Data loaded successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data loaded successfully.


In [ ]:
# Clean NaNs to prevent calculation errors
df_train_clean = df_train[['review/profileName', 'beer/beerId', 'review/overall', 'beer/name', 'beer/style']].dropna()
df_val_clean = df_val[['review/profileName', 'beer/beerId', 'review/overall']].dropna()
df_test_clean = df_test[['review/profileName', 'beer/beerId', 'review/overall']].dropna()

# Calculate popularity statistics
beer_ratings_mean = df_train_clean.groupby('beer/beerId')['review/overall'].mean().reset_index()
beer_ratings_mean.rename(columns={'review/overall': 'average_rating'}, inplace=True)

beer_review_counts = df_train_clean.groupby('beer/beerId').size().reset_index(name='review_count')

# Merge statistics and beer information
beer_popularity = pd.merge(beer_ratings_mean, beer_review_counts, on='beer/beerId')
beer_info = df_train_clean[['beer/beerId', 'beer/name', 'beer/style']].drop_duplicates(subset=['beer/beerId'])
beer_popularity = pd.merge(beer_popularity, beer_info, on='beer/beerId', how='left')

# Filter for top beers based on a minimum review threshold
min_reviews_threshold = 15
popular_high_rated_beers = beer_popularity[beer_popularity['review_count'] >= min_reviews_threshold]
popular_high_rated_beers = popular_high_rated_beers.sort_values(by='average_rating', ascending=False)
print("Data cleaning and popularity calculations complete.")

Data cleaning and popularity calculations complete.


In [ ]:
# Calculate global mean and item means for evaluation predictions
global_mean = df_train_clean['review/overall'].mean()
item_means = df_train_clean.groupby('beer/beerId')['review/overall'].mean().to_dict()

# Function to evaluate the popularity model
def evaluate_popularity(df):
    # Predict the item average; if item is unseen, use the global average
    predictions = df['beer/beerId'].map(item_means).fillna(global_mean)
    rmse = np.sqrt(mean_squared_error(df['review/overall'], predictions))
    mae = mean_absolute_error(df['review/overall'], predictions)
    return rmse, mae

# Evaluate and display metrics
val_rmse, val_mae = evaluate_popularity(df_val_clean)
test_rmse, test_mae = evaluate_popularity(df_test_clean)

print("Popularity Model - Validation Set Performance:")
print(f"RMSE: {val_rmse:.4f} | MAE:  {val_mae:.4f}")
print("\nPopularity Model - Test Set Performance:")
print(f"RMSE: {test_rmse:.4f} | MAE:  {test_mae:.4f}")

print("\n" + "="*50)
print("TOP 10 BEER RECOMMENDATIONS FOR ALL USERS")
print("="*50)
display(popular_high_rated_beers[['beer/name', 'beer/style', 'average_rating', 'review_count']].head(10))

Popularity Model - Validation Set Performance:
RMSE: 0.6208 | MAE:  0.4577

Popularity Model - Test Set Performance:
RMSE: 0.6383 | MAE:  0.4661

TOP 10 BEER RECOMMENDATIONS FOR ALL USERS


,beer/name,beer/style,average_rating,review_count
39201,Rare D.O.S.,American Double / Imperial Stout,4.903846,26
28197,Dirty Horse,Lambic - Unblended,4.838710,31
15837,M Belgian-Style Barleywine,American Barleywine,4.772727,22
41784,Armand'4 Oude Geuze Lente (Spring),Gueuze,4.750000,52
6437,Southampton Berliner Weisse,Berliner Weissbier,4.750000,32
9721,Mühlen Kölsch,Kölsch,4.700000,15
27692,Yellow Bus,American Wild Ale,4.694444,36
14339,Mönchsambacher Lager,Keller Bier / Zwickel Bier,4.675000,20
36380,Hoppy Birthday,American Pale Ale (APA),4.673077,52
13685,Cambridge House Copper Hill Kölsch,Kölsch,4.656250,16
